# 05 · BW + PLI + EM + MA: PEAF, SEAF, PDAF, SDAF (paper Figs. 9, 10, 11, Sec. III-B)

* **PEAF:** parallel EMD adaptive filter (Fig. 4a, Eq. 2)
* **SEAF:** staged EMD adaptive filter (Fig. 4b, Eq. 3)
* **PDAF:** all noises treated as one; one adaptive filter on the whole signal with the summed references
* **SDAF:** Thakor & Zhu [13] plus an MA layer (Fig. 5, no EMD)

PEAF and SEAF are run with both EMD implementations. PDAF and SDAF don't use EMD; under each EMD they run with that
EMD's LMS setting (notebook 02), so the four methods are always compared with the same filter setting.

Paper: **PEAF has the highest SNRimp** on every record at SNRin = 10 dB (Fig. 10) and at every SNRin (Fig. 11), and
the MA filter works better **after** the other three filters than before them (Sec. III-B).

In [ ]:
from pathlib import Path
import sys, json

project_root = Path.cwd() if (Path.cwd() / "src").exists() else Path.cwd().parent
sys.path.insert(0, str(project_root))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
%matplotlib inline

from src.claims import EMDS, EMD_LABEL, PAPER_IMFS
RESULTS = project_root / "results"
plt.rcParams.update({"figure.dpi": 110, "axes.grid": True, "grid.alpha": 0.3})
METHOD_COLORS = {"PEAF": "#1f3b73", "SEAF": "#9aa83a", "PDAF": "#d62728", "SDAF": "#3cb9a0"}
EMD_COLORS = {"matlab": "tab:orange", "pyemd": "tab:purple"}


def show_check(title, result):
    """Print one claim check from src.claims: PASS / PARTLY / FAIL and our numbers."""
    mark = {"Yes": "PASS", "Partly": "PARTLY", "No": "FAIL"}.get(result["verdict"], result["verdict"])
    print(f"[{mark:>6}] {title}\n         {result['ours']}")

In [ ]:
from src import claims

four = pd.read_csv(RESULTS / "four_methods.csv")
print("LMS setting:", json.loads((RESULTS / "lms_setting.json").read_text()))
LABEL = {"PEAF": "proposed emd+adap parallel (PEAF)", "SEAF": "proposed emd+adap staged (SEAF)",
         "PDAF": "adap parallel (PDAF)", "SDAF": "adap stage [13] (SDAF)"}
assert np.isfinite(four[["SNRout_dB", "SNRimp_dB"]].values).all(), "every filter converged"
assert len(four) == 2 * 13 * 6 * 4
print(f"{len(four)} runs, all finite")

## Fig. 9: example (record 100, SNRin = 10 dB, PEAF, first 5000 samples)

In [ ]:
examples = {e: np.load(RESULTS / f"fig9_example_{e}.npz") for e in EMDS}
for e, ex in examples.items():
    fs = int(ex["fs"])
    sigs = [("original signal", ex["clean"]), ("signal + noise", ex["noisy"]), ("EMD denoised signal (PEAF)", ex["PEAF"])]
    fig, axes = plt.subplots(3, 2, figsize=(16, 7.5), gridspec_kw={"width_ratios": [1.6, 1]})
    for (name, s), (ax_t, ax_f) in zip(sigs, axes):
        ax_t.plot(s[:5000], lw=0.6); ax_t.set_title(name, fontsize=10); ax_t.set_ylabel("mV")
        f = np.fft.rfftfreq(len(s), 1 / fs)
        ax_f.plot(f, np.abs(np.fft.rfft(s - s.mean())) / len(s), lw=0.5)
        ax_f.set_xlim(0, 180); ax_f.set_title(name + " (spectrum, whole segment)", fontsize=10)
    axes[-1, 0].set_xlabel("sample"); axes[-1, 1].set_xlabel("f (Hz)")
    fig.suptitle(f"Fig. 9 with {EMD_LABEL[e]}", fontsize=13)
    plt.tight_layout(); plt.show()

### All outputs on the same 3 s window (t = 120 s)

In [ ]:
ex = examples["matlab"]; fs = int(ex["fs"])
w = slice(120 * fs, 123 * fs); t = np.arange(w.start, w.stop) / fs
panels = [(m, e) for m in ["PEAF", "SEAF", "PDAF", "SDAF"] for e in EMDS]
fig, axes = plt.subplots(len(panels), 1, figsize=(14, 18), sharex=True)
for ax, (m, e) in zip(axes, panels):
    ex = examples[e]
    ax.plot(t, ex["noisy"][w], color="0.75", lw=0.6, label="signal + noise")
    ax.plot(t, ex["clean"][w], color="k", lw=1.2, label="original")
    ax.plot(t, ex[m][w], color=METHOD_COLORS[m], lw=0.9, label=m)
    ax.set_ylabel("mV"); ax.legend(fontsize=8, loc="upper right")
    ax.set_title(LABEL[m] + (f", {EMD_LABEL[e]}" if m in ("PEAF", "SEAF") else f", {EMD_LABEL[e]}'s LMS setting"), fontsize=10)
axes[-1].set_xlabel("time (s)")
plt.tight_layout(); plt.show()

## Fig. 10: SNRimp per record, SNRin = 10 dB

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(18, 4.8), sharey=True)
tables = {}
for ax, e in zip(axes, EMDS):
    d10 = four[(four.emd == e) & (four.SNRin_dB == 10)].pivot(index="record", columns="method", values="SNRimp_dB")
    for m in ["PEAF", "SEAF", "PDAF", "SDAF"]:
        ax.plot([str(r) for r in d10.index], d10[m], marker="o", color=METHOD_COLORS[m], label=LABEL[m])
    ax.set_xlabel("signal"); ax.set_ylabel("SNRimp (dB)"); ax.legend(fontsize=8)
    ax.set_title(f"{EMD_LABEL[e]}: SNRimp using different signal (SNRin = 10 dB)")
    t10 = d10.copy(); t10["best"] = d10.idxmax(axis=1); t10.loc["mean"] = d10.mean()
    tables[EMD_LABEL[e]] = t10
plt.tight_layout(); plt.show()
for e in EMDS:
    show_check(f"Fig. 10, {EMD_LABEL[e]}: PEAF best on every record", claims.peaf_best_per_record(four, e))
pd.concat(tables, axis=1).round(2)

## Fig. 11: SNRimp vs SNRin (mean over the 13 records)

Black crosses: the paper's PEAF line, read off its Fig. 11 (about ±0.5 dB).

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(17, 9), gridspec_kw={"width_ratios": [1.3, 1]})
tables = {}
for row, e in enumerate(EMDS):
    d = four[four.emd == e]
    g = d.groupby(["SNRin_dB", "method"]).SNRimp_dB
    mean11 = g.mean().unstack()
    ax = axes[row, 0]
    for m in ["PEAF", "SEAF", "PDAF", "SDAF"]:
        ax.plot([str(s) for s in mean11.index], mean11[m], marker="o", color=METHOD_COLORS[m], label=LABEL[m])
    ax.plot([str(s) for s in claims.PAPER_FIG11_PEAF], list(claims.PAPER_FIG11_PEAF.values()), "kx", ms=9,
            mew=2, label="PEAF, paper's Fig. 11")
    ax.set_xlabel("SNRinput (dB)"); ax.set_ylabel("SNRimp (dB)"); ax.legend(fontsize=8)
    ax.set_title(f"{EMD_LABEL[e]}: SNRimp under different SNRinput (mean of 13 records)")
    data = [d[(d.SNRin_dB == s) & (d.method == m)].SNRimp_dB.values
            for s in mean11.index for m in ["PEAF", "SEAF", "PDAF", "SDAF"]]
    pos = [i * 5 + k for i in range(len(mean11)) for k in range(4)]
    bp = axes[row, 1].boxplot(data, positions=pos, widths=0.8, patch_artist=True, showfliers=False)
    for patch, k in zip(bp["boxes"], [k for _ in mean11.index for k in range(4)]):
        patch.set_facecolor(list(METHOD_COLORS.values())[k])
    axes[row, 1].set_xticks([i * 5 + 1.5 for i in range(len(mean11))], mean11.index)
    axes[row, 1].set_xlabel("SNRinput (dB)"); axes[row, 1].set_ylabel("SNRimp (dB)")
    axes[row, 1].set_title(f"{EMD_LABEL[e]}: spread over the 13 records")
    out = mean11.copy(); out["best"] = mean11.idxmax(axis=1)
    tables[EMD_LABEL[e]] = out
plt.tight_layout(); plt.show()
pd.concat(tables, axis=1).round(2)

In [ ]:
for e in EMDS:
    show_check(f"Fig. 11, {EMD_LABEL[e]}: PEAF best at every SNRin", claims.peaf_best_per_snr(four, e))
    show_check(f"Fig. 11, {EMD_LABEL[e]}: SNRimp falls as SNRin rises", claims.snrimp_falls(four, e))
    show_check(f"Fig. 11, {EMD_LABEL[e]}: PEAF within 2 dB of the paper", claims.fig11_magnitude(four, e))

## Sec. III-B: MA filter after the other three, or before?

*"The adaptive filter for denoising MA should be placed either before or after denoising the other three noises ...
For denoising MA, we found that setting the adaptive filter after denoising the other three has a better performance."*

* **after** (the paper's structures): PEAF / SEAF as in Fig. 4, MA filter last.
* **before**: the MA filter runs on the noisy ECG first; EMD is then applied to its output and the PLI, BW and EM
  filters of PEAF / SEAF run on those IMFs. SDAF is shown too (MA filter first or last, no EMD).

In [ ]:
ma = pd.read_csv(RESULTS / "ma_order.csv")
assert np.isfinite(ma.SNRimp_dB).all() and len(ma) == 2 * 13 * 6 * 3 * 2
# the "after" runs are the same as the four-method results
chk = ma[ma.ma_filter == "after"].merge(four, on=["emd", "record", "SNRin_dB", "method"])
assert np.allclose(chk.SNRimp_dB_x, chk.SNRimp_dB_y)

fig, axes = plt.subplots(1, 2, figsize=(17, 4.8), sharey=True)
for ax, e in zip(axes, EMDS):
    d = ma[ma.emd == e].groupby(["method", "ma_filter", "SNRin_dB"]).SNRimp_dB.mean()
    for m in ["PEAF", "SEAF", "SDAF"]:
        for pos, ls in [("after", "-"), ("before", "--")]:
            v = d[m][pos]
            ax.plot([str(s) for s in v.index], v.values, ls, marker="o", color=METHOD_COLORS[m], label=f"{m}, MA filter {pos}")
    ax.set_xlabel("SNRinput (dB)"); ax.set_ylabel("SNRimp (dB)"); ax.legend(fontsize=8)
    ax.set_title(f"{EMD_LABEL[e]}: MA filter after vs before the other three")
plt.tight_layout(); plt.show()
for e in EMDS:
    show_check(f"Sec. III-B, {EMD_LABEL[e]}: MA filter after is better", claims.ma_after_better(ma, e))
ma.pivot_table(index=["emd", "method"], columns=["ma_filter"], values="SNRimp_dB", aggfunc="mean").round(3)